# Semantic Segmentation Training Boilerplate

**Purpose:** A flexible, configurable notebook for training binary/multi-class segmentation models using PyTorch and `segmentation_models_pytorch` (SMP).

**Features:**
- Easily switch between different architectures (UNet, UNet++, DeepLabV3+, etc.)
- Choose any SMP-compatible encoder (ResNet, EfficientNet, etc.) with pretrained weights
- Configurable loss functions (Combo, Tversky, Focal Tversky, etc.)
- Progressive training with loss switching (optional)
- Early stopping, learning rate scheduling, and best model checkpointing
- Full training/validation/test evaluation with Dice and IoU metrics
- Visualization of predictions

**How to use:**
1. Set your dataset paths and hyperparameters in the **Configuration** cell.
2. Ensure your dataset follows the expected folder structure (see below).
3. Run all cells.
4. The best model will be saved automatically.

**Expected dataset structure:**

```
any_dataset/
├── images/          # .jpg
|	├── train/
|	|	├── train_001.jpg
|	|	└── ...
|	├── test/
|	|	├── test_001.jpg
|	|	└── ...
|	└── val/
|		├── val_001.jpg
|		└── ...
└── masks/           # .png, 0 = black = background, 255 = white = crack
	├── train/
	|	├── train_001.png
	|	└── ...
	├── test/
	|	├── test_001.png
	|	└── ...
	└── val/
		├── val_001.png
		└── ...
```


Masks should be single-channel with pixel values:
- For binary: 0 = background, 255 = foreground (or any >0).
- For multi-class: integer values 0, 1, 2, ...

The code will automatically binarize masks if needed (threshold > 0) but you can adapt.

## Notebook Setup & Dependencies

Install necessary packages and import core libraries.

In [ ]:
# Cell 1: Install required packages (if not already installed)
!pip install -q segmentation-models-pytorch opencv-python-headless albumentations onnxscript --quiet

In [ ]:
# Cell 2: Core Imports and Seed Setting
import os
import cv2
import csv
import torch
import onnxscript     # For Exporting
import numpy as np
import random
import zipfile
import warnings
from pathlib import Path
from dataclasses import dataclass, field  # For configuration
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score # For evaluation

warnings.filterwarnings('ignore')

# Mount Google Drive (Initially for Path Inspection - Recommended)
from google.colab import drive
drive.mount('/content/drive')

def set_seed(seed: int = 42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # If CUDA available
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("Libraries imported and seeds set.")

## Centralized Configuration
Define all hyperparameters, paths, model settings, etc., in one place using a @dataclass. This makes customization easy.

In [ ]:
# Cell 3
@dataclass
class Config:
    # ---------- DATA PATHS ----------
    ROOT_PROJECT_PATH: str = "/path/to/your/project"  # make sure it points at rooot
    DATA_TRANSPARENCY: str = "public"                 # e.g., "public" or "private"
    DATA_NAME: str = "YourDatasetName"
    ZIPPED_DATA_PATH: str = f"{ROOT_PROJECT_PATH}/datasets/{DATA_TRANSPARENCY}_images/{DATA_NAME}.zip"
    EXTRACT_TO: str = "/content"                      # Or another suitable path

    # ---------- FILE EXTENSIONS ----------
    IMAGE_EXT: tuple = ('.jpg', '.jpeg', '.png')
    MASK_EXT: tuple = ('.png', '.jpg', '.jpeg', '.bmp')

    # ---------- TRAINING PARAMETERS ----------
    IMAGE_HEIGHT: int = 576 # Target height after resizing
    IMAGE_WIDTH: int = 800  # Target width after resizing
    BATCH_SIZE: int = 8
    EPOCHS: int = 100       # Set based on your needs
    LEARNING_RATE: float = 1e-4
    NUM_WORKERS: int = 0    # Set to 0 if facing multiprocessing issues
    VAL_SPLIT: float = 0.2  # Fraction for validation
    TEST_SPLIT: float = 0.0 # Fraction for test (or use existing test folders)

    # ---------- MODEL & SMP ----------
    MODEL_NAME: str = "Unet"          # Options: Unet, UnetPlusPlus, DeepLabV3Plus, FPN, PAN, Linknet
    ENCODER_NAME: str = "resnet34"    # e.g., resnet34, efficientnet-b0, mit_b0
    ENCODER_WEIGHTS: str = "imagenet" # Pretrained weights
    IN_CHANNELS: int = 3              # Usually 3 for RGB
    NUM_CLASSES: int = 1              # 1 for binary, N for N-class

    # ---------- LOSS & SCHEDULER ----------
    LOSS_TYPE: str = "dice"       # Options: dice, bce, tversky, focal_tversky, combo
    SCHEDULER_PATIENCE: int = 5   # Patience for ReduceLROnPlateau
    SCHEDULER_FACTOR: float = 0.5 # Factor for ReduceLROnPlateau

    # ---------- OUTPUT ----------
    # OUTPUT_DIR: str = f"./{MODEL_NAME}_{DATA_TRANSPARENCY}-{DATA_NAME}_outputs"
    OUTPUT_DIR: str = f"./{MODEL_NAME}_{DATA_NAME}_outputs"
    DEVICE: str = field(init=False) # Will be set automatically

    def __post_init__(self):
        self.DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Instantiate configuration
config = Config()

print(f"Configuration Loaded:")
print(f"  Device: {config.DEVICE}")
print(f"  Model: {config.MODEL_NAME} + {config.ENCODER_NAME}")
print(f"  Loss: {config.LOSS_TYPE}")
print(f"  Data ZIP: {config.ZIPPED_DATA_PATH}")
print(f"  Output Dir: {config.OUTPUT_DIR}")

## Data Loading & Preparation

Handle dataset extraction (if zipped), locate image/mask pairs, split data, and define the custom PyTorch Dataset.

In [ ]:
# Cell 4: Data Extraction and Discovery

def extract_data(zip_path, extract_to):
    """Extracts a zip file to a specified directory."""
    print(f"Extracting {zip_path} to {extract_to}...")
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Extraction done.")
    return extract_to

def find_files(root_dir, extensions):
    """Finds files with given extensions recursively."""
    files = []
    for ext in extensions:
        files.extend(Path(root_dir).rglob(f"*{ext}"))
        files.extend(Path(root_dir).rglob(f"*{ext.upper()}")) # Also check uppercase
    return sorted(set(files))

# Extract data if path is provided and file exists
if Path(config.ZIPPED_DATA_PATH).is_file():
    extracted_path = extract_data(config.ZIPPED_DATA_PATH, config.EXTRACT_TO)
else:
    extracted_path = config.ROOT_PROJECT_PATH # Assume data is already unzipped at ROOT_PROJECT_PATH

# Locate images and masks directories
dataset_root = None
for candidate in Path(extracted_path).iterdir():
    if candidate.is_dir() and (candidate / "images").exists() and (candidate / "masks").exists():
        dataset_root = candidate
        break

if dataset_root is None:
    # Check if images/masks are directly in extracted_path
    if (Path(extracted_path) / "images").exists() and (Path(extracted_path) / "masks").exists():
        dataset_root = Path(extracted_path)
    else:
        raise FileNotFoundError("Could not find 'images' and 'masks' folders.")

images_dir = dataset_root / "images"
masks_dir = dataset_root / "masks"

print(f"Images directory: {images_dir}")
print(f"Masks directory: {masks_dir}")

# Find all image and mask paths
all_images = find_files(images_dir, config.IMAGE_EXT)
all_masks = []
for img_path in all_images:
    stem = img_path.stem
    candidates = list(Path(masks_dir).rglob(f"{stem}.*"))
    candidates = [p for p in candidates if p.suffix.lower() in config.MASK_EXT]
    if candidates:
        all_masks.append(candidates[0])
    else:
        print(f"Warning: No mask found for image {img_path}")

image_paths = [str(p) for p in all_images]
mask_paths = [str(p) for p in all_masks]

assert len(image_paths) == len(mask_paths), f"Mismatch between images ({len(image_paths)}) and masks ({len(mask_paths)})"

print(f"Found {len(image_paths)} image-mask pairs.")

In [ ]:
# Cell 5: Handle Test Split (Check for pre-existing test folders)

test_images_dir = dataset_root / "images" / "test"
test_masks_dir = dataset_root / "masks" / "test"

if test_images_dir.exists() and test_masks_dir.exists():
    print("Found pre-existing 'test' folders. Using them.")
    test_image_paths = find_files(test_images_dir, config.IMAGE_EXT)
    test_mask_paths = []
    for img_path in test_image_paths:
        stem = img_path.stem
        candidates = list(Path(test_masks_dir).rglob(f"{stem}.*"))
        candidates = [p for p in candidates if p.suffix.lower() in config.MASK_EXT]
        if candidates:
            test_mask_paths.append(candidates[0])
        else:
            print(f"Warning: No test mask found for image {img_path}")
    test_image_paths = [str(p) for p in test_image_paths]
    test_mask_paths = [str(p) for p in test_mask_paths]

    # Split remaining data into train/val
    remaining_image_paths = [p for p in image_paths if p not in test_image_paths]
    remaining_mask_paths = [p for p in mask_paths if p not in test_mask_paths]
    train_val_img, val_img, train_val_mask, val_mask = train_test_split(
        remaining_image_paths, remaining_mask_paths, test_size=config.VAL_SPLIT, random_state=42
    )
    train_img, train_mask = train_val_img, train_val_mask
else:
    print("No pre-existing 'test' folders found. Splitting data according to TEST_SPLIT and VAL_SPLIT.")
    # Split full dataset into train/val/test
    train_val_img, test_img, train_val_mask, test_mask = train_test_split(
        image_paths, mask_paths, test_size=config.TEST_SPLIT if config.TEST_SPLIT > 0 else 0.0, random_state=42
    )
    train_img, val_img, train_mask, val_mask = train_test_split(
        train_val_img, train_val_mask, test_size=config.VAL_SPLIT, random_state=42
    )

print(f"Train samples: {len(train_img)}, Validation samples: {len(val_img)}, Test samples: {len(test_img)}")


In [ ]:
# Cell 6: Dataset Class Definition

class SegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, target_height, target_width, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.target_h = target_height
        self.target_w = target_width
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load Image
        img = cv2.imread(self.image_paths[idx])
        if img is None:
             raise FileNotFoundError(f"Image could not be loaded: {self.image_paths[idx]}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert BGR to RGB

        # Load Mask
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        if mask is None:
             raise FileNotFoundError(f"Mask could not be loaded: {self.mask_paths[idx]}")
        # For binary segmentation, convert mask to 0/1 (0=background, 1=foreground)
        # Original code used (mask > 0).astype(np.float32), which is correct for binary.
        # For multi-class, ensure mask values are integers 0, 1, 2, ..., C-1
        mask = (mask > 0).astype(np.float32) # Assumes binary

        # Resize manually before applying transforms
        img = cv2.resize(img, (self.target_w, self.target_h), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (self.target_w, self.target_h), interpolation=cv2.INTER_NEAREST) # Nearest for masks

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']

        return img, mask

def get_transforms():
    """Define augmentation transforms (without resize, as it's done manually in the dataset)."""
    train_transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        # Avoid A.RandomRotate90 if resizing afterwards due to potential dimension mismatches
        A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.2, rotate_limit=15, p=0.5),
        A.OneOf([
            A.GridDistortion(num_steps=5, distort_limit=0.05, p=1.0),
            A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=1.0)
        ], p=0.25),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.75),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # ImageNet stats
        ToTensorV2(),
    ])

    valid_transform = A.Compose([
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # ImageNet stats
        ToTensorV2(),
    ])

    return train_transform, valid_transform

# Get transforms
train_transform, valid_transform = get_transforms()

# Create Datasets
train_dataset = SegmentationDataset(train_img, train_mask, config.IMAGE_HEIGHT, config.IMAGE_WIDTH, transform=train_transform)
val_dataset = SegmentationDataset(val_img, val_mask, config.IMAGE_HEIGHT, config.IMAGE_WIDTH, transform=valid_transform)
test_dataset = SegmentationDataset(test_image_paths, test_mask_paths, config.IMAGE_HEIGHT, config.IMAGE_WIDTH, transform=valid_transform) # Use valid_transform for test

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=config.NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)

## Model & Loss Definition

Provide functions to instantiate the SMP model and select the loss function based on configuration.

In [ ]:
# Cell 7: Model and Loss Functions
def create_model(cfg):
    """Creates and returns an SMP model based on configuration."""
    model_map = {
        "Unet": smp.Unet,
        "UnetPlusPlus": smp.UnetPlusPlus,
        "DeepLabV3Plus": smp.DeepLabV3Plus,
        "FPN": smp.FPN,
        "PAN": smp.PAN,
        "Linknet": smp.Linknet,
    }

    if cfg.MODEL_NAME not in model_map:
        raise ValueError(f"Model {cfg.MODEL_NAME} not supported by this boilerplate.")

    model = model_map[cfg.MODEL_NAME](
        encoder_name=cfg.ENCODER_NAME,
        encoder_weights=cfg.ENCODER_WEIGHTS,
        in_channels=cfg.IN_CHANNELS,
        classes=cfg.NUM_CLASSES,
    )
    return model.to(cfg.DEVICE)

def get_loss_function(loss_type, mode='binary'):
    """Returns an SMP loss function based on type."""
    if loss_type == 'dice':
        return smp.losses.DiceLoss(mode=mode)
    elif loss_type == 'bce':
        return smp.losses.BCEWithLogitsLoss() # BCEWithLogits is often preferred over BCE
    elif loss_type == 'tversky':
        return smp.losses.TverskyLoss(mode=mode, alpha=0.7, beta=0.3) # Example params
    elif loss_type == 'focal_tversky':
        return smp.losses.FocalTverskyLoss(mode=mode, alpha=0.7, beta=0.3, gamma=0.75) # Example params
    elif loss_type == 'combo':
        return smp.losses.ComboLoss(mode=mode, alpha=0.5, beta=0.5) # Example params
    else:
        raise ValueError(f"Unknown loss type: {loss_type}")

# Example usage within training loop
# model = create_model(config)
# loss_fn = get_loss_function(config.LOSS_TYPE, mode='binary' if config.NUM_CLASSES == 1 else 'multiclass')

## Training & Evaluation Logic

Define the core training and validation loops, including metric calculation.

In [ ]:
# Cell 8: Metrics Calculation and Training/Validation Loops

def calculate_metrics(pred, true, threshold=0.5):
    """Calculate IoU, Dice, Precision, Recall, F1 for a batch."""
    pred_sig = torch.sigmoid(pred) if config.NUM_CLASSES == 1 else torch.softmax(pred, dim=1) # Handle multi-class
    pred_bin = (pred_sig > threshold).float()

    # Ensure masks are also float32 and have the right shape for broadcasting
    true = true.float().unsqueeze(1) if config.NUM_CLASSES == 1 and true.ndim != pred_bin.ndim else true.float()

    # Calculate Intersection and Union for IoU/Dice
    intersection = (pred_bin * true).sum(dim=(1, 2, 3))
    union = (pred_bin + true).sum(dim=(1, 2, 3)) - intersection

    iou = (intersection + 1e-6) / (union + 1e-6) # Add small epsilon
    dice = (2.0 * intersection + 1e-6) / (pred_bin.sum(dim=(1, 2, 3)) + true.sum(dim=(1, 2, 3)) + 1e-6)

    # Flatten tensors for sklearn metrics (only works reliably for batch_size=1 or aggregated later)
    # For aggregated batch metrics, we calculate them element-wise and then average.
    # Let's aggregate over the batch dimension for these metrics too for consistency.
    true_flat_batch = true.flatten(start_dim=1) # Shape: (batch_size, H*W*C or H*W)
    pred_flat_batch = pred_bin.flatten(start_dim=1) # Shape: (batch_size, H*W*C or H*W)

    # Calculate metrics per item in the batch, then average
    precisions = []
    recalls = []
    f1s = []
    for i in range(true_flat_batch.shape[0]):
        true_flat_item = true_flat_batch[i].cpu().numpy()
        pred_flat_item = pred_flat_batch[i].cpu().numpy()
        # Use average='macro' or 'weighted' for multiclass if needed
        p = precision_score(true_flat_item, pred_flat_item, average='binary', zero_division=0)
        r = recall_score(true_flat_item, pred_flat_item, average='binary', zero_division=0)
        f1 = f1_score(true_flat_item, pred_flat_item, average='binary', zero_division=0)
        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)

    precision = np.mean(precisions)
    recall = np.mean(recalls)
    f1 = np.mean(f1s)

    return iou.mean(), dice.mean(), precision, recall, f1


def train_epoch(model, dataloader, optimizer, loss_fn, device):
    """Performs one training epoch."""
    model.train()
    total_loss, total_iou, total_dice, total_precision, total_recall, total_f1 = 0, 0, 0, 0, 0, 0

    for images, masks in tqdm(dataloader, desc="Training"):
        images = images.to(device, dtype=torch.float32)
        masks = masks.to(device, dtype=torch.float32) # Already float from dataset

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks.unsqueeze(1) if config.NUM_CLASSES == 1 else masks) # Adjust mask shape if needed
        loss.backward()
        optimizer.step()

        iou, dice, precision, recall, f1 = calculate_metrics(outputs, masks)
        total_loss += loss.item()
        total_iou += iou.item()
        total_dice += dice.item()
        total_precision += precision
        total_recall += recall
        total_f1 += f1

    return (
        total_loss / len(dataloader),
        total_iou / len(dataloader),
        total_dice / len(dataloader),
        total_precision / len(dataloader),
        total_recall / len(dataloader),
        total_f1 / len(dataloader)
    )

def validate_epoch(model, dataloader, loss_fn, device):
    """Performs one validation epoch."""
    model.eval()
    total_loss, total_iou, total_dice, total_precision, total_recall, total_f1 = 0, 0, 0, 0, 0, 0

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Validating"):
            images = images.to(device, dtype=torch.float32)
            masks = masks.to(device, dtype=torch.float32)

            outputs = model(images)
            loss = loss_fn(outputs, masks.unsqueeze(1) if config.NUM_CLASSES == 1 else masks) # Adjust mask shape if needed

            iou, dice, precision, recall, f1 = calculate_metrics(outputs, masks)
            total_loss += loss.item()
            total_iou += iou.item()
            total_dice += dice.item()
            total_precision += precision
            total_recall += recall
            total_f1 += f1

    return (
        total_loss / len(dataloader),
        total_iou / len(dataloader),
        total_dice / len(dataloader),
        total_precision / len(dataloader),
        total_recall / len(dataloader),
        total_f1 / len(dataloader)
    )

In [ ]:
# Cell 9: Main Training Function

def train_model(cfg, train_loader, val_loader):
    """Main training loop with checkpointing and history tracking."""
    model = create_model(cfg)
    loss_fn = get_loss_function(cfg.LOSS_TYPE, mode='binary' if cfg.NUM_CLASSES == 1 else 'multiclass')
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=cfg.SCHEDULER_PATIENCE, factor=cfg.SCHEDULER_FACTOR, verbose=True
    )

    best_val_loss = float('inf')
    best_val_metric_key = 'val_iou' # Or 'val_dice' depending on preference
    best_val_metric_value = -float('inf') # Track best metric value
    history = {
        'train_loss': [], 'val_loss': [],
        'train_iou': [], 'val_iou': [],
        'train_dice': [], 'val_dice': [],
        'train_precision': [], 'val_precision': [],
        'train_recall': [], 'val_recall': [],
        'train_f1': [], 'val_f1': []
    }

    for epoch in range(cfg.EPOCHS):
        print(f"\nEpoch {epoch+1}/{cfg.EPOCHS}")
        print("-" * 50)

        train_results = train_epoch(model, train_loader, optimizer, loss_fn, cfg.DEVICE)
        val_results = validate_epoch(model, val_loader, loss_fn, cfg.DEVICE)

        train_loss, train_iou, train_dice, train_precision, train_recall, train_f1 = train_results
        val_loss, val_iou, val_dice, val_precision, val_recall, val_f1 = val_results

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_iou'].append(train_iou)
        history['val_iou'].append(val_iou)
        history['train_dice'].append(train_dice)
        history['val_dice'].append(val_dice)
        history['train_precision'].append(train_precision)
        history['val_precision'].append(val_precision)
        history['train_recall'].append(train_recall)
        history['val_recall'].append(val_recall)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)

        print(f"Train Loss: {train_loss:.4f}, IoU: {train_iou:.4f}, Dice: {train_dice:.4f}, Prec: {train_precision:.4f}, Rec: {train_recall:.4f}, F1: {train_f1:.4f}")
        print(f"Val Loss: {val_loss:.4f}, IoU: {val_iou:.4f}, Dice: {val_dice:.4f}, Prec: {val_precision:.4f}, Rec: {val_recall:.4f}, F1: {val_f1:.4f}")

        # Checkpoint based on validation loss (or another metric like IoU/Dice)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            checkpoint_path = Path(cfg.OUTPUT_DIR) / f"{cfg.MODEL_NAME}_{cfg.ENCODER_NAME}_best_loss.pth"
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_iou': val_iou,
                'val_dice': val_dice,
                'val_precision': val_precision,
                'val_recall': val_recall,
                'val_f1': val_f1,
            }, checkpoint_path)
            print(f"New best model (loss) saved at epoch {epoch+1}")

        # Optional: Checkpoint based on best metric (e.g., IoU)
        if val_iou > best_val_metric_value:
            best_val_metric_value = val_iou
            checkpoint_path_metric = Path(cfg.OUTPUT_DIR) / f"{cfg.MODEL_NAME}_{cfg.ENCODER_NAME}_best_{best_val_metric_key}.pth"
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_iou': val_iou,
                'val_dice': val_dice,
                'val_precision': val_precision,
                'val_recall': val_recall,
                'val_f1': val_f1,
            }, checkpoint_path_metric)
            print(f"New best model ({best_val_metric_key}) saved at epoch {epoch+1}")

        scheduler.step(val_loss) # Adjust LR based on validation loss

    # Plot metrics after training
    plot_training_history(history, cfg.OUTPUT_DIR)

    return model, history

def plot_training_history(history, output_dir):
    """Plots training and validation metrics."""
    plt.figure(figsize=(18, 10)) # Adjusted for multiple plots

    plt.subplot(2, 3, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(2, 3, 2)
    plt.plot(history['train_iou'], label='Train IoU')
    plt.plot(history['val_iou'], label='Val IoU')
    plt.title('IoU')
    plt.xlabel('Epoch')
    plt.ylabel('IoU')
    plt.legend()

    plt.subplot(2, 3, 3)
    plt.plot(history['train_dice'], label='Train Dice')
    plt.plot(history['val_dice'], label='Val Dice')
    plt.title('Dice')
    plt.xlabel('Epoch')
    plt.ylabel('Dice')
    plt.legend()

    plt.subplot(2, 3, 4)
    plt.plot(history['train_precision'], label='Train Precision')
    plt.plot(history['val_precision'], label='Val Precision')
    plt.title('Precision')
    plt.xlabel('Epoch')
    plt.ylabel('Precision')
    plt.legend()

    plt.subplot(2, 3, 5)
    plt.plot(history['train_recall'], label='Train Recall')
    plt.plot(history['val_recall'], label='Val Recall')
    plt.title('Recall')
    plt.xlabel('Epoch')
    plt.ylabel('Recall')
    plt.legend()

    plt.subplot(2, 3, 6)
    plt.plot(history['train_f1'], label='Train F1')
    plt.plot(history['val_f1'], label='Val F1')
    plt.title('F1 Score')
    plt.xlabel('Epoch')
    plt.ylabel('F1')
    plt.legend()

    plt.tight_layout()
    plot_path = Path(output_dir) / "training_metrics_plot.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"Training metrics plot saved to {plot_path}")
    plt.show() # Or plt.close() if running headless

## Training Execution

Call the main training function with configured parameters.

In [ ]:
# Cell 10: Start Training

# Note: You can override config settings here if needed before training
# config.EPOCHS = 50
# config.MODEL_NAME = "DeepLabV3Plus"
# config.LOSS_TYPE = "focal_tversky"

print("Starting Training...")
model, training_history = train_model(config, train_loader, val_loader)
print("Training Completed.")

## Results Saving

Purpose: Save training history (CSV, JSON) and aggregated metrics.

In [ ]:
# Cell 11: Save Detailed Training History

history_path_csv = Path(config.OUTPUT_DIR) / "training_history.csv"
with open(history_path_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['epoch', 'train_loss', 'val_loss', 'train_iou', 'val_iou',
                     'train_dice', 'val_dice', 'train_precision', 'val_precision',
                     'train_recall', 'val_recall', 'train_f1', 'val_f1'])
    for epoch in range(len(training_history['train_loss'])):
        writer.writerow([
            epoch + 1,
            training_history['train_loss'][epoch],
            training_history['val_loss'][epoch],
            training_history['train_iou'][epoch],
            training_history['val_iou'][epoch],
            training_history['train_dice'][epoch],
            training_history['val_dice'][epoch],
            training_history['train_precision'][epoch],
            training_history['val_precision'][epoch],
            training_history['train_recall'][epoch],
            training_history['val_recall'][epoch],
            training_history['train_f1'][epoch]
        ])
print(f"Detailed training history saved to {history_path_csv}")

# Cell 12: Save Aggregated Metrics
aggregated_metrics = {}
for key in training_history:
    aggregated_metrics[f"mean_{key}"] = np.mean(training_history[key])

aggregated_metrics_path = Path(config.OUTPUT_DIR) / "training_metrics.json"
with open(aggregated_metrics_path, 'w') as f:
    json.dump(aggregated_metrics, f, indent=4)
print(f"Aggregated training metrics saved to {aggregated_metrics_path}")

## Test Evaluation

Evaluate the final (best) model on the test set and save results.

In [ ]:
# Cell 14: Evaluate on Test Set

def evaluate_model_on_dataloader(model, dataloader, device, loss_fn, threshold=0.5):
    """Evaluates the model on a given dataloader (e.g., test set)."""
    model.eval()
    total_loss = 0
    all_iou, all_dice = [], []
    all_true_flat, all_pred_flat = [], []

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Evaluating on test set"):
            images = images.to(device, dtype=torch.float32)
            masks = masks.to(device, dtype=torch.float32) # Already float

            outputs = model(images)
            loss = loss_fn(outputs, masks.unsqueeze(1) if config.NUM_CLASSES == 1 else masks) # Adjust mask shape
            total_loss += loss.item()

            pred_sig = torch.sigmoid(outputs) if config.NUM_CLASSES == 1 else torch.softmax(outputs, dim=1)
            pred_bin = (pred_sig > threshold).float()

            # Calculate IoU/Dice for this batch
            intersection = (pred_bin * masks.unsqueeze(1)).sum(dim=(1, 2, 3)) # Ensure mask has channel dim
            union = (pred_bin + masks.unsqueeze(1)).sum(dim=(1, 2, 3)) - intersection
            iou_batch = (intersection + 1e-6) / (union + 1e-6)
            dice_batch = (2.0 * intersection + 1e-6) / (pred_bin.sum(dim=(1, 2, 3)) + masks.unsqueeze(1).sum(dim=(1, 2, 3)) + 1e-6)

            all_iou.extend(iou_batch.cpu().numpy())
            all_dice.extend(dice_batch.cpu().numpy())

            # Flatten for sklearn metrics (accumulate over batches)
            all_true_flat.extend(masks.flatten().cpu().numpy())
            all_pred_flat.extend(pred_bin.flatten().cpu().numpy())

    # Calculate overall metrics
    avg_loss = total_loss / len(dataloader)
    avg_iou = np.mean(all_iou)
    avg_dice = np.mean(all_dice)

    precision_overall = precision_score(all_true_flat, all_pred_flat, average='binary', zero_division=0)
    recall_overall = recall_score(all_true_flat, all_pred_flat, average='binary', zero_division=0)
    f1_overall = f1_score(all_true_flat, all_pred_flat, average='binary', zero_division=0)

    print("\n=== Test Set Evaluation ===")
    print(f"  Average Loss: {avg_loss:.4f}")
    print(f"  Mean IoU:     {avg_iou:.4f}")
    print(f"  Mean Dice:    {avg_dice:.4f}")
    print(f"  Overall Prec: {precision_overall:.4f}")
    print(f"  Overall Rec:  {recall_overall:.4f}")
    print(f"  Overall F1:   {f1_overall:.4f}")

    return {
        'test_loss': avg_loss,
        'test_iou': avg_iou,
        'test_dice': avg_dice,
        'test_precision': precision_overall,
        'test_recall': recall_overall,
        'test_f1': f1_overall
    }

# Load the best model (based on loss or chosen metric)
checkpoint_path = Path(config.OUTPUT_DIR) / f"{config.MODEL_NAME}_{config.ENCODER_NAME}_best_loss.pth" # Or _best_iou.pth etc.
# checkpoint_path = Path(config.OUTPUT_DIR) / f"{config.MODEL_NAME}_{config.ENCODER_NAME}_best_val_iou.pth"
if not checkpoint_path.exists():
    print(f"Warning: Best model checkpoint not found at {checkpoint_path}. Attempting to load from training end state.")
    # If no checkpoint was saved, the 'model' variable holds the final state
    print("Using final training model state for evaluation.")
else:
    print(f"Loading best model from {checkpoint_path}")
    model = create_model(config) # Re-initialize
    checkpoint = torch.load(checkpoint_path, map_location=config.DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(config.DEVICE) # Ensure model is on correct device

# Define loss function for evaluation
eval_loss_fn = get_loss_function(config.LOSS_TYPE, mode='binary' if config.NUM_CLASSES == 1 else 'multiclass')

# Run evaluation
test_metrics = evaluate_model_on_dataloader(model, test_loader, config.DEVICE, eval_loss_fn)

# Save test metrics
test_metrics_csv_path = Path(config.OUTPUT_DIR) / "test_metrics.csv"
with open(test_metrics_csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['test_loss', 'test_iou', 'test_dice', 'test_precision', 'test_recall', 'test_f1'])
    writer.writerow([test_metrics[k] for k in ['test_loss', 'test_iou', 'test_dice', 'test_precision', 'test_recall', 'test_f1']])
print(f"Test metrics saved to {test_metrics_csv_path}")

test_metrics_json_path = Path(config.OUTPUT_DIR) / "test_metrics.json"
with open(test_metrics_json_path, 'w') as f:
    json.dump(test_metrics, f, indent=4)
print(f"Test metrics JSON saved to {test_metrics_json_path}")

## Visualization

Visualize model predictions against ground truth on test samples.

In [ ]:
# Cell 16: Visualize Predictions on Test Samples
def visualize_predictions_test(model, dataloader, device, n_samples=4, save_path=None):
    """Visualizes model predictions on a few test samples."""
    model.eval()
    fig, axes = plt.subplots(3, n_samples, figsize=(15, 10))
    if n_samples == 1:
        axes = axes[:, np.newaxis] # Ensure axes indexing works correctly

    with torch.no_grad():
        for i, (images, masks) in enumerate(dataloader):
            if i >= n_samples:
                break
            images = images.to(device)
            masks = masks.numpy() # Convert to numpy for plotting

            outputs = model(images)
            preds = torch.sigmoid(outputs).cpu().numpy() if config.NUM_CLASSES == 1 else torch.softmax(outputs, dim=1)[:, 1:, :, :].cpu().numpy() # For binary, take prob of class 1

            # Denormalize image for display
            img = images.cpu().permute(0, 2, 3, 1).numpy()[0] # Take first image in batch
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            img = np.clip(img * std + mean, 0, 1)

            axes[0, i].imshow(img)
            axes[0, i].set_title("Input Image")
            axes[0, i].axis('off')

            axes[1, i].imshow(masks[0], cmap='gray') # Take first mask in batch
            axes[1, i].set_title("Ground Truth")
            axes[1, i].axis('off')

            axes[2, i].imshow(preds[0, 0], cmap='gray') # Take first prediction in batch, first channel (prob of foreground)
            axes[2, i].set_title("Prediction")
            axes[2, i].axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Test predictions visualization saved to {save_path}")
    plt.show()

# Create a small dataloader for visualization (shuffle=False to get consistent samples)
vis_test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
vis_save_path = Path(config.OUTPUT_DIR) / "test_predictions_visualization.png"
visualize_predictions_test(
    model, vis_test_loader, config.DEVICE,
    n_samples=min(4, len(test_dataset)),
    save_path=vis_save_path
)